In [1]:
from pathlib import Path
from zipfile import ZipFile
from io import TextIOWrapper
from gensim import corpora
from gensim.models import LdaModel

from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

import spacy
import re
import random
import pyLDAvis
import pyLDAvis.gensim_models

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Chargement des données

Nous ne travaillons que sur les données de 1993.

In [2]:
zip_path = Path("../data/legislatives_1993.zip")
print(zip_path.exists())

True


In [3]:

texts = []

with ZipFile(zip_path, 'r') as zf:
    for name in zf.namelist():

        # On ne garde que les vrais fichiers .txt (évite les dossiers)
        if name.lower().endswith(".txt"):
            try:
                with zf.open(name) as f:
                    try:
                        txt = TextIOWrapper(f, encoding="utf-8", errors="replace").read()
                    except UnicodeDecodeError:
                        # fallback latin-1
                        f.seek(0)
                        txt = TextIOWrapper(f, encoding="latin-1", errors="replace").read()

                texts.append(txt)

            except Exception as e:
                print(f" Erreur sur {name} : {e}")

print(f"{len(texts)} articles chargés")

5936 articles chargés


In [4]:
# Premier article
print(len(texts[0]))
print(texts[0]) 
print("********************************************")


5527
Département de Seine-Maritime - 12ème Circonscription - Scrutin du 21 Mars 1993
Alain LE VERN Député Maire de Saint-Saëns - 44 ans Suppléant : Docteur Christian PLAILLY Maire de GAILLEFONTAINE - 44 ans
Chère Madame, Chère Mademoiselle, Cher Monsieur,
Voici 5 ans vous m'avez élu Député. J'ai depuis consacré tout mon temps et toute mon énergie pour être digne de votre confiance. La période électorale est trop souvent celle des divisions, des belles promesses ... Jugez les actes, les faits ! Nous n'avons pas tout réussi, bien sûr, mais j'ai travaillé pour rassembler et unir dans l'intérêt de notre circonscription, de ses habitants, de notre Pays.
DANS NOTRE CIRCONSCRIPTION, J'AI AGI POUR: ☒ La construction de logements (Gournay, La Feuillie, Buchy, Saint-Saëns, Neuville Ferrières, Neufchâtel, Londinières, Blangy, Torcy le Grand, Bellencombre ... une centaine par an au lieu d'une dizaine par an avant !). ☒ Le désenclavement routier : RN 27 Rouen Dieppe, Autoroute A 28 ouverte en décem

In [5]:
# Deuxième article
print(len(texts[1]))
print(texts[1])
print("********************************************")

4293
ELECTIONS LEGISLATIVES DU 21 MARS 1993
REPUBLIQUE FRANÇAISE - 2™e CIRCONSCRIPTION DE LA DORDOGNE
Michel SUCHOD Député du Bergeracois
Suppléant
François LASTERNAS Conseiller général de La Force Maire de Prigonrieux
DEUX SOCIALISTES POUR LA RELEVE DE LA GAUCHE DEUX HOMMES EFFICACES POUR LE BERGERACOIS
Madame, Mademoiselle, Monsieur,
Comme vous je suis citoyen.
Et si comme vous, je pense que le gouvernement sortant ne rend pas copie blanche, je pense comme vous qu'il n'a pas su régler le dramatique problème de l'emploi, et qu'il a laissé se développer des « affaires » trop nombreuses et particulièrement nauséabondes.
Il faut donc changer la politique, pour recréer l'emploi, raffermir la démocratie, sauver la paix.
Cela implique-t-il de liquider le député sortant du Bergeracois pour le remplacer par la représen- tante de la droite conservatrice comme certains le souhaitent ? Poser la question, c'est y répondre par la négative.
Bien mieux, il faut garder Michel SUCHOD, le député qui :


# Nettoyage

In [6]:

def clean_text(text):
    
    text = text.lower()
    
    # supprimer mentions archives
    text = re.sub(r"sciences po / fonds cevipof", "", text)
    
    # supprimer caractères spéciaux
    text = re.sub(r"[☒•«»]", " ", text)
    
    # supprimer chiffres
    text = re.sub(r"\d+", " ", text)
    
    # supprimer ponctuation
    text = re.sub(r"[^\w\s]", " ", text)
    
    # supprimer espaces multiples
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

clean_texts = [clean_text(t) for t in texts]
print(clean_texts[0])

département de seine maritime ème circonscription scrutin du mars alain le vern député maire de saint saëns ans suppléant docteur christian plailly maire de gaillefontaine ans chère madame chère mademoiselle cher monsieur voici ans vous m avez élu député j ai depuis consacré tout mon temps et toute mon énergie pour être digne de votre confiance la période électorale est trop souvent celle des divisions des belles promesses jugez les actes les faits nous n avons pas tout réussi bien sûr mais j ai travaillé pour rassembler et unir dans l intérêt de notre circonscription de ses habitants de notre pays dans notre circonscription j ai agi pour la construction de logements gournay la feuillie buchy saint saëns neuville ferrières neufchâtel londinières blangy torcy le grand bellencombre une centaine par an au lieu d une dizaine par an avant le désenclavement routier rn rouen dieppe autoroute a ouverte en décembre dernier jusqu à neufchâtel rn aménagée une formation de meilleure qualité adapta

In [7]:
# récupérer 30 documents aléatoires

clean_texts_aleatoires = random.sample(clean_texts, 30)

# LDA topic modeling

In [8]:
!python -m spacy download fr_core_news_md

/opt/python/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=1843) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 90.4 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_md')


In [9]:
nlp = spacy.load("fr_core_news_md")

In [10]:
def preprocess(text):
    
    doc = nlp(text)
    
    tokens = [
        token.lemma_
        for token in doc
        if token.is_alpha
        and not token.is_stop
        and len(token) > 3
    ]
    
    return tokens

processed_texts = [preprocess(t) for t in clean_texts_aleatoires]
processed_texts[0]

['république',
 'français',
 'département',
 'circonscription',
 'élection',
 'législatif',
 'mars',
 'entente',
 'écologiste',
 'vert',
 'génération',
 'écologie',
 'marc',
 'antoni',
 'professeur',
 'économie',
 'conseiller',
 'municipal',
 'bourg',
 'marier',
 'enfant',
 'crise',
 'écologique',
 'crise',
 'social',
 'cause',
 'sortir',
 'falloir',
 'trouver',
 'solution',
 'progrès',
 'innovation',
 'social',
 'ensemble',
 'choisir',
 'avenir',
 'vivre',
 'présent',
 'choisir',
 'audace',
 'oser',
 'écologie',
 'départ',
 'écologie',
 'sembler',
 'rêve',
 'réalité',
 'donner',
 'raison',
 'écologiste',
 'réanimer',
 'démocratie',
 'préserver',
 'planète',
 'permettre',
 'humain',
 'vivre',
 'dignité',
 'urgence',
 'aujourd',
 'écologie',
 'inspirer',
 'projet',
 'monde',
 'moderne',
 'nouveau',
 'manière',
 'aborder',
 'difficulté',
 'société',
 'chômage',
 'récession',
 'économique',
 'solitude',
 'absence',
 'solidarité',
 'fraternité',
 'écologie',
 'vouloir',
 'réconcilier',
 'é

In [11]:
# Corpus LDA

dictionary = corpora.Dictionary(processed_texts)

dictionary.filter_extremes(
    no_below=5,      # mot doit apparaître dans 5 docs
    no_above=0.4     # supprimer mots trop fréquents
)

corpus = [dictionary.doc2bow(text) for text in processed_texts]

# Modèle LDA
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=10,
    passes=15,
    random_state=42
)

In [12]:
# voir les topics
topics = lda_model.print_topics(num_words=10)

for t in topics:
    print(t)

(0, '0.051*"nature" + 0.051*"animal" + 0.040*"rassemblement" + 0.030*"choix" + 0.026*"cher" + 0.021*"département" + 0.021*"etat" + 0.021*"activité" + 0.021*"bulletin" + 0.020*"rassembler"')
(1, '0.049*"travailleur" + 0.036*"parti" + 0.027*"ouvrier" + 0.020*"unité" + 0.020*"école" + 0.018*"peuple" + 0.017*"million" + 0.017*"milliard" + 0.016*"gouvernement" + 0.016*"fonds"')
(2, '0.017*"devoir" + 0.017*"honnêteté" + 0.017*"affaire" + 0.017*"agir" + 0.017*"sortir" + 0.017*"engagement" + 0.017*"alternance" + 0.017*"économie" + 0.017*"progrès" + 0.017*"neuf"')
(3, '0.040*"confiance" + 0.026*"progrès" + 0.020*"monsieur" + 0.020*"action" + 0.020*"participation" + 0.020*"électeur" + 0.020*"canton" + 0.020*"responsabilité" + 0.019*"face" + 0.019*"général"')
(4, '0.019*"région" + 0.016*"écologiste" + 0.013*"écologie" + 0.013*"entente" + 0.011*"formation" + 0.011*"tour" + 0.010*"créer" + 0.010*"économie" + 0.009*"besoin" + 0.009*"agriculture"')
(5, '0.031*"immigré" + 0.028*"immigration" + 0.025*"

In [13]:
# Visualisations


vis = pyLDAvis.gensim_models.prepare(
    lda_model,
    corpus,
    dictionary
)

pyLDAvis.display(vis)

# BERTopic

In [29]:
import re

def clean_for_bertopic(text):
    text = text.lower()

    # mentions d’archive / impression
    patterns = [
        r"sciences po\s*/\s*fonds cevipof",
        r"sciences po\s*/\s*fonds cevipov",
        r"fonds cevipof",
        r"fonds cevipov",
        r"vu[,]?\s*les?\s*candidats?",
        r"vu[,]?\s*le\s*candidat",
        r"imp\.[^\n]*",
        r"offset[^\n]*",
    ]
    for p in patterns:
        text = re.sub(p, " ", text)

    # symboles parasites
    text = re.sub(r"[☒☐•▪■◆●«»“”„+]", " ", text)

    # chiffres isolés
    text = re.sub(r"\b\d+\b", " ", text)

    # ponctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # espaces multiples
    text = re.sub(r"\s+", " ", text).strip()

    return text

docs_clean = [clean_for_bertopic(t) for t in texts]



In [15]:
!pip install -U bertopic sentence-transformers umap-learn hdbscan scikit-learn

/opt/python/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=1843) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


In [16]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cuda"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3009.39it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# # calcul des embedings
# embeddings = embedding_model.encode(
#     docs,
#     batch_size=64,
#     show_progress_bar=True,
#     convert_to_numpy=True,
#     normalize_embeddings=True
# )

# print(embeddings.shape)   # attendu: (5936, 384)

Batches: 100%|██████████| 93/93 [00:14<00:00,  6.44it/s]

(5936, 384)


In [18]:
## Sauvegarde des embeddings
# import numpy as np
# np.save("embeddings_1993.npy", embeddings)

In [30]:

nlp = spacy.load("fr_core_news_md")

stopwords = nlp.Defaults.stop_words

extra_stopwords = [
    "france", "français", "francaise", "francais",
    "circonscription", "elections", "election", "legislatives", "legislative",
    "mars", "ans", "candidat", "candidats", "candidate", "suppléant", "suppléante",
    "votez", "voter", "vu", "madame", "monsieur", "mademoiselle",
    "notre", "nos", "votre", "vos", "je", "nous", "vous",
    "politique", "pays", "contre", "droit", "droite", "gauche",
    "po", "fonds", "cevipof", "cevipov"
]

stopwords = list(nlp.Defaults.stop_words) + extra_stopwords

In [33]:
# pipeline BERTopic



umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=120,
    min_samples=15,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words=stopwords,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.75
)

topic_model = BERTopic(
    embedding_model=None,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    min_topic_size=120,
    calculate_probabilities=True,
    verbose=True
)



In [34]:
docs_clean = [clean_for_bertopic(t) for t in texts]

embeddings = embedding_model.encode(
    docs_clean,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

topics, probs = topic_model.fit_transform(docs_clean, embeddings)

topic_info = topic_model.get_topic_info()
topic_info.head(20)

Batches: 100%|██████████| 93/93 [00:15<00:00,  5.91it/s]
2026-03-14 19:41:16,259 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-14 19:41:31,317 - BERTopic - Dimensionality - Completed ✓
2026-03-14 19:41:31,319 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-14 19:41:31,748 - BERTopic - Cluster - Completed ✓
2026-03-14 19:41:31,752 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-14 19:41:36,864 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,311,-1_loi naturelle_naturelle_parti travailleurs_...,"[loi naturelle, naturelle, parti travailleurs,...",[elections legislatives mars votre candidat du...
1,0,4293,0_rpr_udf_national_rpr udf,"[rpr, udf, national, rpr udf, voulez, front, f...",[élections législatives de mars 12ème circonsc...
2,1,777,1_entente cologistes_eclogie cologie_choix_eco...,"[entente cologistes, eclogie cologie, choix, e...",[elections législatives mars troisième circons...
3,2,362,2_marseille rpublique_animauxvotez_nature anim...,"[marseille rpublique, animauxvotez, nature ani...",[république française élections législatives d...
4,3,193,3_bourgeoisie_qu prenne_emplois salaires_compt...,"[bourgeoisie, qu prenne, emplois salaires, com...",[elections législatives du mars pourquoi voter...


In [26]:
len(topic_info)

19

In [27]:
# Lire les mots d'un topic
for topic_id in topic_info["Topic"].head(10):
    if topic_id != -1:
        print(f"\n=== Topic {topic_id} ===")
        print(topic_model.get_topic(topic_id)[:10])


=== Topic 0 ===
[('franais', np.float64(0.016249125241655293)), ('politique', np.float64(0.013884342517590622)), ('cest', np.float64(0.012305552804507196)), ('droite', np.float64(0.011599225775281868)), ('contre', np.float64(0.010394520076253468)), ('pays', np.float64(0.010386789546918972)), ('nationale', np.float64(0.010048541131008193)), ('sociale', np.float64(0.00979342168120028)), ('dput', np.float64(0.009718733534709052)), ('rpr', np.float64(0.008882256599709315))]

=== Topic 1 ===
[('lcologie', np.float64(0.025400644881601403)), ('verts', np.float64(0.021661859630162004)), ('ecologistes', np.float64(0.019879317493164638)), ('vie', np.float64(0.01730306069082342)), ('lentente', np.float64(0.014854857312598203)), ('ecologie', np.float64(0.013065273099305854)), ('lentente ecologistes', np.float64(0.012600212404770298)), ('politique', np.float64(0.012151648671072771)), ('entente', np.float64(0.012035782733451019)), ('generation', np.float64(0.01162328568267462))]

=== Topic 2 ===
[(

In [28]:
new_topics, new_probs = topic_model.reduce_topics(
    docs,
    topics=topics,
    nr_topics=15
)

topic_info_reduced = topic_model.get_topic_info()
topic_info_reduced.head(20)

TypeError: BERTopic.reduce_topics() got an unexpected keyword argument 'topics'. Did you mean 'nr_topics'?

In [ ]:
fig1 = topic_model.visualize_topics()
fig1.show()
fig1.write_html("fig_intertopic_map.html")

In [ ]:
fig2 = topic_model.visualize_barchart(top_n_topics=12, n_words=8)
fig2.show()
fig2.write_html("fig_topic_barchart.html")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

topic_info_plot = topic_model.get_topic_info()
topic_info_plot = topic_info_plot[topic_info_plot["Topic"] != -1].copy()
topic_info_plot = topic_info_plot.sort_values("Count", ascending=False).head(15)

plt.figure(figsize=(10, 5))
plt.bar(topic_info_plot["Name"], topic_info_plot["Count"])
plt.xticks(rotation=70, ha="right")
plt.ylabel("Nombre de documents")
plt.title("Taille des principaux topics")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

df_topics = pd.DataFrame({
    "doc_id": range(len(docs)),
    "text": docs,
    "topic": topics,
    "topic_prob": [float(np.max(p)) if p is not None else np.nan for p in probs]
})

df_topics.head()